In [1]:
!nvidia-smi

Thu Oct 29 20:43:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/road_damage', exist_ok=True)
print("Drive mounted")

Mounted at /content/drive
Drive mounted


In [3]:
!pip install ultralytics roboflow albumentations -q

import torch
print(f"✅ Torch: {torch.__version__}")
print(f"✅ CUDA:  {torch.cuda.is_available()}")
print(f"✅ GPU:   {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

✅ Torch: 2.11.0+cu130
✅ CUDA:  True
✅ GPU:   Tesla T4


In [4]:
from roboflow import Roboflow
rf = Roboflow(api_key="3gKLYe7kpiIiO6m0gVgw")
project = rf.workspace("nours-workspace-scvhz").project("road-damage-l1ju7-pgtvw")
version = project.version(2)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to road-damage-2 in yolov8:: 100%|██████████| 13518/13518 [00:02<00:00, 4584.61it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [5]:
import yaml, os

yaml_path = "/content/road-damage-2/data.yaml"

with open(yaml_path, "r") as f:
    config = yaml.safe_load(f)

base = "/content/road-damage-2"
config["train"] = f"{base}/train/images"
config["val"]   = f"{base}/valid/images"
config["test"]  = f"{base}/test/images"

with open(yaml_path, "w") as f:
    yaml.dump(config, f)

print("Paths fixed!")
print("Classes:", config["names"])
print("NC:     ", config["nc"])

Paths fixed!
Classes: ['alligator', 'block', 'crack', 'edge', 'longitudinal', 'pothole', 'transverse']
NC:      7


In [6]:
import os
import json
import time
import numpy as np
from datetime import datetime
from ultralytics import YOLO

In [22]:
DATA_YAML       = "/content/road-damage-2/data.yaml"
EXPERIMENTS_DIR = "/content/content/experiments"
OUTPUTS_DIR     = "/content/content/outputs"
RESULTS_JSON    = "/content/outputs/results.json"
PLOTS_DIR       = "/content/outputs/plots"
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR,     exist_ok=True)
os.makedirs(PLOTS_DIR,       exist_ok=True)


In [23]:
MODELS      = ["yolov5su.pt", "yolov8s.pt", "yolo11s.pt"]
EPOCHS      = 30
BATCH_SIZE  = 16
IMAGE_SIZE  = 640
OPTIMIZER   = "AdamW"
SEED        = 42

In [24]:
def train_one_model(weights_name: str) -> dict:
    model_name = weights_name.replace(".pt", "")
    run_name   = f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    print(f"\n{'='*55}")
    print(f"  Training: {model_name}")
    print(f"{'='*55}")

    model = YOLO(weights_name)
    model.train(
        data          = DATA_YAML,
        epochs        = EPOCHS,
        batch         = BATCH_SIZE,
        imgsz         = IMAGE_SIZE,
        optimizer     = OPTIMIZER,
        lr0           = 0.001,
        lrf           = 0.01,
        momentum      = 0.937,
        weight_decay  = 0.0005,
        warmup_epochs = 3,
        mosaic        = 1.0,
        mixup         = 0.1,
        copy_paste    = 0.1,
        fliplr        = 0.5,
        hsv_h         = 0.015,
        hsv_s         = 0.7,
        hsv_v         = 0.4,
        close_mosaic  = 10,
        patience      = 20,
        seed          = SEED,
        project       = EXPERIMENTS_DIR,
        name          = run_name,
        device        = 0,
        verbose       = True,
        exist_ok      = True,
    )

    best_weights_path = os.path.join(EXPERIMENTS_DIR, run_name, "weights", "best.pt")
    best_model = YOLO(best_weights_path)
    metrics    = best_model.val(data=DATA_YAML, verbose=False)

    map50     = round(float(metrics.box.map50), 4)
    map50_95  = round(float(metrics.box.map),   4)
    precision = round(float(metrics.box.mp),    4)
    recall    = round(float(metrics.box.mr),    4)

    class_names = ['alligator','block','crack','edge','longitudinal','pothole','transverse']
    per_class   = {}
    if hasattr(metrics.box, 'ap_class_index') and metrics.box.ap_class_index is not None:
        for i, cls_idx in enumerate(metrics.box.ap_class_index):
            cls_name = class_names[cls_idx] if cls_idx < len(class_names) else str(cls_idx)
            per_class[cls_name] = {
                "AP50":      round(float(metrics.box.ap50[i]),  4),
                "AP50_95":   round(float(metrics.box.ap[i]),    4),
            }
    size_mb = round(os.path.getsize(best_weights_path) / 1e6, 2)

    dummy = np.zeros((640, 640, 3), dtype=np.uint8)
    for _ in range(10):
        best_model.predict(dummy, verbose=False, device=0)

    N  = 100
    t0 = time.perf_counter()
    for _ in range(N):
        best_model.predict(dummy, verbose=False, device=0)
    elapsed = time.perf_counter() - t0

    fps    = round(N / elapsed,        1)
    ms_img = round((elapsed / N)*1000, 2)

    result = {
        "model_name":        model_name,
        "run_name":          run_name,
        "best_weights_path": best_weights_path,
        "mAP50":             map50,
        "mAP50-95":          map50_95,
        "precision":         precision,
        "recall":            recall,
        "model_size_mb":     size_mb,
        "fps":               fps,
        "ms_per_image":      ms_img,
        "per_class":         per_class,
    }

    print(f"\n {model_name} done!")
    print(f"   mAP@0.5:      {map50}")
    print(f"   mAP@0.5:0.95: {map50_95}")
    print(f"   Precision:    {precision}")
    print(f"   Recall:       {recall}")
    print(f"   Size:         {size_mb} MB")
    print(f"   FPS:          {fps}")
    print(f"   ms/image:     {ms_img}")

    return result


In [25]:
def train_all_models():
    results = []
    for weights_name in MODELS:
        result = train_one_model(weights_name)
        results.append(result)
    return results


In [26]:
def print_comparison_table(results: list):
    print(f"\n{'='*75}")
    print("  MODEL COMPARISON TABLE")
    print(f"{'='*75}")
    header = f"{'Model':<12}{'mAP50':<10}{'mAP50-95':<12}{'Precision':<12}{'Recall':<10}{'Size(MB)':<11}{'FPS':<8}"
    print(header)
    print("-" * 75)
    for r in results:
        print(
            f"{r['model_name']:<12}"
            f"{r['mAP50']:<10}"
            f"{r['mAP50-95']:<12}"
            f"{r['precision']:<12}"
            f"{r['recall']:<10}"
            f"{r['model_size_mb']:<11}"
            f"{r['fps']:<8}"
        )
    print(f"{'='*75}")

    best = max(results, key=lambda r: r["mAP50-95"])
    print(f"\n Best model: {best['model_name']}  (mAP50-95={best['mAP50-95']})")
    print(f"   Weights: {best['best_weights_path']}")
    print(f"\n{'='*75}")
    print("  PER-CLASS AP@0.5 BREAKDOWN")
    print(f"{'='*75}")
    class_names = ['alligator','block','crack','edge','longitudinal','pothole','transverse']
    print(f"{'Class':<15}" + "".join(f"{r['model_name']:<14}" for r in results))
    print("-" * 75)
    for cls in class_names:
        row = f"{cls:<15}"
        for r in results:
            val = r['per_class'].get(cls, {}).get('AP50', 'N/A')
            row += f"{str(val):<14}"
        print(row)

    return best


In [27]:
def save_results(results: list):
    os.makedirs(os.path.dirname(RESULTS_JSON), exist_ok=True)
    with open(RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\n Results saved to: {RESULTS_JSON}")


In [28]:
all_results = train_all_models()
best_model  = print_comparison_table(all_results)
save_results(all_results)


  Training: yolov5su
Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/road-damage-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov5su.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov5su_20261029_215303, nbs=64,

In [29]:
import os
import glob

# Check content directory first
print("=== Files in /content/ ===")
for root, dirs, files in os.walk("/content"):
    # Skip drive and sample_data
    dirs[:] = [d for d in dirs if d not in ['drive', 'sample_data', '.config']]
    for f in files:
        if f.endswith('.pt'):
            full_path = os.path.join(root, f)
            size_mb   = os.path.getsize(full_path) / 1e6
            print(f"  ✅ {full_path}  ({size_mb:.1f} MB)")

# Check Drive
print("\n=== Files in Drive ===")
for root, dirs, files in os.walk("/content/drive/MyDrive/road_damage"):
    for f in files:
        if f.endswith('.pt'):
            full_path = os.path.join(root, f)
            size_mb   = os.path.getsize(full_path) / 1e6
            print(f"  ✅ {full_path}  ({size_mb:.1f} MB)")

=== Files in /content/ ===
  ✅ /content/yolov8s.pt  (22.6 MB)
  ✅ /content/yolov5su.pt  (18.6 MB)
  ✅ /content/yolo11s.pt  (19.3 MB)
  ✅ /content/content/experiments/yolov5su_20261029_215303/weights/best.pt  (18.5 MB)
  ✅ /content/content/experiments/yolov5su_20261029_215303/weights/last.pt  (18.5 MB)
  ✅ /content/content/experiments/yolov5su_20261029_214821/weights/best.pt  (91.7 MB)
  ✅ /content/content/experiments/yolov5su_20261029_214821/weights/last.pt  (91.7 MB)
  ✅ /content/content/experiments/yolo11s_20261029_233142/weights/best.pt  (19.2 MB)
  ✅ /content/content/experiments/yolo11s_20261029_233142/weights/last.pt  (19.2 MB)
  ✅ /content/content/experiments/yolov8s_20261029_224049/weights/best.pt  (22.5 MB)
  ✅ /content/content/experiments/yolov8s_20261029_224049/weights/last.pt  (22.5 MB)
  ✅ /content/runs/detect/content/experiments/yolov5su_20261029_205857/weights/best.pt  (18.5 MB)
  ✅ /content/runs/detect/content/experiments/yolov5su_20261029_205857/weights/last.pt  (18.5 M

In [ ]:
import shutil, os
weights = {
    "yolov5su": "/content/content/experiments/yolov5su_20261029_215303/weights/best.pt",
    "yolov8s":  "/content/content/experiments/yolov8s_20261029_224049/weights/best.pt",
    "yolo11s":  "/content/content/experiments/yolo11s_20261029_233142/weights/best.pt",
}

drive_dir = "/content/drive/MyDrive/road_damage/weights"
os.makedirs(drive_dir, exist_ok=True)

for model_name, src in weights.items():
    dst = f"{drive_dir}/{model_name}_best.pt"
    print(f"Copying {model_name}...")
    shutil.copy2(src, dst)
    size = os.path.getsize(dst) / 1e6
    print(f"  ✅ {dst}  ({size:.1f} MB)")

print("\n All weights saved to Drive!")

Copying yolov5su...
  ✅ /content/drive/MyDrive/road_damage/weights/yolov5su_best.pt  (18.5 MB)
Copying yolov8s...
  ✅ /content/drive/MyDrive/road_damage/weights/yolov8s_best.pt  (22.5 MB)
Copying yolo11s...
  ✅ /content/drive/MyDrive/road_damage/weights/yolo11s_best.pt  (19.2 MB)

✅ All weights saved to Drive!


In [ ]:
from google.colab import files

for model_name, src in weights.items():
    local = f"/content/{model_name}_best.pt"
    shutil.copy2(src, local)
    print(f"  Downloading {model_name}_best.pt ...")
    files.download(local)

⬇️  Downloading yolov5su_best.pt ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading yolov8s_best.pt ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading yolo11s_best.pt ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>